# Inpainting using INR (Implicit Neural Representations)

## Introduction

In recent years, artificial intelligence has become increasingly effective at processing images and videos – not only recognizing objects but also **completing missing parts of an image**, **compressing data**, or even **generating new video frames**.

Imagine you have a video where part of the image – for example, a rectangular fragment in each frame – has been obscured, as if someone cut out that part with scissors or covered it with tape. Your task is to **reconstruct the missing fragments based solely on pixel coordinates and time**.

In this task, we will focus on a more advanced problem: creating a model of an entire video as a function that, for any pixel coordinates and frame number, returns its color and object belonging, using **inpainting** techniques and **INR networks**.

### What is inpainting?

**Inpainting** is a technique used in image and video processing that involves **filling in missing fragments** of an image or video in the most realistic way possible. The name comes from English and literally means "painting in". It is used, among other things, for removing objects from photos, reconstructing damaged photographs, or repairing damaged video frames.

Example use of inpainting:

![](https://live.staticflickr.com/65535/54554097818_7395667a21.jpg)

In our task, we will work on a more advanced case: **dynamic inpainting**, i.e., filling in **missing video fragments over time**. The missing parts occur only in a selected area (e.g., the center of the frame) that has been cut out in all frames of the video. Importantly, the model does not have to generate the entire frame – its task is **only to predict the content of the missing region**.

### What are INR (Implicit Neural Representations)?

**INR (Implicit Neural Representations)** is a modern technique for representing data using neural networks. Traditionally, images or videos are stored as matrices (grids) of pixel values – each cell of such a matrix is the color of a pixel. However, INRs work differently: instead of storing pixel values directly, the network learns a **function** that, for given spatial coordinates (x, y) and – in our case – also temporal (t), returns the predicted values.

So instead of storing the entire video, we train a network that acts like a "virtual projector":

$$
f(x, y, t) \rightarrow (R, G, B)
$$

Where:

* $x, y$ – pixel coordinates,
* $t$ – video frame number (time),
* $R, G, B$ – pixel color at that location,

Such representation has many advantages:

* allows generating images at any resolution (we are not limited by a specific pixel grid),
* enables interpolation between frames (e.g., creating smooth motion),
* works well in tasks such as compression, super-resolution, or inpainting.

This happens because the network does not limit us to integer values; we can choose any intermediate values, e.g., (1.5, 44.5, 13.25).

An example use of INR networks is shown in the graphic:

![](https://live.staticflickr.com/65535/54555506837_48f1f8e589_b.jpg)

In our task, the INR model will represent only the **missing video fragment**, i.e., the area we do not know. The network must learn not only **how the image looks over time** but also **what is in the invisible area**. This means it must recognize the context of the scene – based on the surroundings – and on that basis realistically reconstruct the missing data (the *inpainting* task).

### What is a segmentation mask?

The second part of the problem concerns **image segmentation**. Segmentation is the process of recognizing which pixels belong to an object (e.g., a character, car, tree) and which to the background. This can be written in the form of a **mask** – an image (in our case, for simplicity, we will use a binary mask) in which each pixel has a value of 0 (does not belong to the object) or 1 (belongs to the object). This allows, for example, separating a character from the background, which is useful in many applications – from autonomous vehicles to video editing.

The graphic shows an example binary mask:

![](https://live.staticflickr.com/65535/54554043929_118739686d_b.jpg)

The designed model will be tasked with predicting **both the pixel color** and its **class (object/background)** – and only for the missing area. Therefore, the above function can be extended to the form:

$$
f(x, y, t) \rightarrow (R, G, B, m)
$$

Where:

* $x, y$ – pixel coordinates,
* $t$ – video frame number (time),
* $R, G, B$ – pixel color at that location,
* $m \in \{0,1\}$ – segmentation mask value.

## Task

Your task is to build an **INR** (Implicit Neural Representation) network that takes as input three numbers:

* **x** – horizontal pixel coordinate,
* **y** – vertical pixel coordinate,
* **t** – video frame number.

> *Note: Input values do not have to be integers – the network should also work for continuous coordinates, which enables interpolation.*

The network output should predict:

* **RGB values** – i.e., the pixel color at that moment in the video,
* **segmentation mask value** – a value of 0 or 1 indicating pixel belonging to the object (0 – background, 1 – object).

### Data

We provide two datasets for the task:

* **Training set** – contains point coordinates and corresponding RGB values and masks, intended for model training,
* **Validation set** – used to evaluate the quality of model predictions on previously unseen data.

For convenience, we have prepared a ready-made **dataloader** that provides data in records containing:

* pixel coordinates: `(x, y, t)`,
* RGB value of the original image,
* segmentation mask value from the set `{0, 1}`.

The input images have a resolution of **256 × 256 pixels**. The datasets contain respectively:

* **training set** – 59 images,
* **validation set** – 10 images,
* **test set** – 10 images.

In the validation set, only pixel coordinates belonging to the **missing area** are provided – this region (size **64 × 64**) is to be *reconstructed* by the model. This means the model should learn to fill in the missing fragments based on previously learned data.

Your solution will ultimately be tested on the Competition Platform on a hidden test dataset, which does not differ significantly from the validation set in terms of data distribution.

### Evaluation Criteria

As you might expect, in the evaluation we will assess two key aspects of your solution:

- **Quality of the missing image region reconstruction** - the quality of the returned RGB pixel area and its coherence, evaluated using the *PSNR* metric,
- **Accuracy of binary mask prediction** - how well the mask values in the missing area were predicted, evaluated using classification *accuracy*.

Definition: **PSNR** (*peak signal-to-noise ratio*) - a popular metric for reconstruction quality (e.g., of an image).

The *PSNR* and *acc* values are averaged over the entire test set.

The final score is defined as the weighted average of these two aspects:

$$ score = 0.7 \cdot P_{PSNR} \cdot 10 + 0.3 \cdot P_{acc} \cdot 10 $$

where $P_{PSNR}$ is the points awarded for the quality result (*PSNR metric value*) of the solution according to thresholds:

$$ P_{PSNR} = \begin{cases}
0 & \text{if } PSNR < 14 \\
2 & \text{if } 14 <= PSNR < 17.1 \\
\frac{5}{4} \cdot PSNR - 19.375 & \text{if } 17.1 <= PSNR < 23.5 \\
10 & \text{if } 23.5 <= PSNR
\end{cases} $$

and $P_{acc}$ is the points awarded for mask accuracy (*acc metric value*) according to thresholds:

$$ P_{acc} = \begin{cases}
0 & \text{if } acc < 0.83 \\
\frac{20}{3} \cdot acc - \frac{83}{15} & \text{if } 0.83 <= acc < 0.98 \\
10 & \text{if } 0.98 <= acc
\end{cases} $$

This formula indicates that to earn points, your solution must achieve a minimum *PSNR* of $14$ or a minimum *acc* of $0.83$, and the maximum number of points ($100$) is awarded for solutions with *PSNR* values from $23.5$ (inclusive) and *acc* values from $0.98$ (inclusive).

> **Note: Models based on standard activations like ReLU may not be sufficient.** \
> *A simple network with ReLU activations achieves only about 5 points of reconstruction quality (PSNR) on the validation set – to achieve significantly better results, consider using sinusoidal activations according to the SIREN approach (Implicit Neural Representations with Periodic Activation Functions), https://arxiv.org/abs/2006.09661.*

Details of implementing this formula can be found in the `grade` function in the task code.

## Constraints

* Your solution will be tested on the Competition Platform.
* The model **cannot** use other datasets or pre-trained weights on other datasets.
* The model can be trained for **a maximum** of 6 minutes using GPU.

## Submission Files

This notebook completed with your solution (see class `YourSolution`).

## Evaluation

Remember that during checking, the `FINAL_EVALUATION_MODE` flag will be set to `True`.

You can earn between 0 and 100 points for this task. The number of points you will receive will be calculated on the (secret) test set on the Competition Platform based on the above formula, rounded to an integer. If your solution does not meet the above criteria or does not execute correctly, you will receive 0 points for the task.

The graphic below illustrates an example inpainting process using an INR network:

![](https://live.staticflickr.com/65535/54554043974_246b208130_b.jpg)

## Starter Code

In this section, we initialize the environment by importing the necessary libraries and functions. The prepared code will make it easier for you to efficiently operate on data and build a proper solution.

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

# During evaluation of your solution, the FINAL_EVALUATION_MODE flag will be set to True
FINAL_EVALUATION_MODE = False

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

import json
import os
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from skimage.metrics import peak_signal_noise_ratio
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import tempfile
import tarfile

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

# Setting the random seed to ensure deterministic results.

seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

### Loading Data

The code below loads the data.

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

class ImageDataset(Dataset):
    """
    The ImageDataset class represents a dataset.
    It handles images and corresponding masks, and generates coordinates
    for each pixel of the image along with a time value (t_value).
    """

    # Number of frames in the video
    VIDEO_LENGTH = 79

    def __init__(
        self,
        img_dir: Path,
        mask_dir: Path,
        frame_names: list[str],
        mode: str,
        json_path: str | None = None,
    ):
        """
        Initializes an instance of the ImageDataset class.

        Parameters:
        ----------
        img_dir : Path
            Path to the directory with images.
        mask_dir : str
            Path to the directory with masks.
        frame_names : list[str]
            List of image file names (without paths).
        mode : str
            Dataset mode ("train" or "val").
        json_path : str | None
            Path to the JSON file with rectangle coordinates (default None).
        """
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.frame_names = frame_names
        self.mode = mode

        if mode not in ["train", "val"]:
            raise ValueError("Invalid mode. Use 'train' or 'val'.")

        if mode == "val" and json_path is None:
            raise ValueError(
                "In 'val' mode, you must provide the path to the JSON file with rectangle coordinates."
            )

        if self.mode == "val":
            # Load missing rectangle coordinates from JSON file
            with open(json_path, "r") as f:
                self.rect_coords = json.load(f)

    def __len__(self) -> int:
        """Returns the number of frames in the dataset.

        Returns:
            Number of frames: int
        """
        return len(self.frame_names)

    def __getitem__(self, idx: int) -> tuple:
        """
        Returns data for the given index.

        Parameters
        ----------
        idx : int
            Frame index.

        Returns
        -------
        Dict[str, Any]
            Dictionary containing depending on mode; (x, y, t),
            images, masks, and other data.
        """
        # Get frame name
        frame_name = self.frame_names[idx]
        img_name = frame_name
        mask_name = frame_name.replace(".jpg", ".png")

        # Load image and mask
        img = cv2.imread(os.path.join(self.img_dir, img_name))
        mask = cv2.imread(os.path.join(self.mask_dir, mask_name), cv2.IMREAD_GRAYSCALE)

        if self.mode == "train":
            # Load rectangle coordinates and normalize
            h, w = img.shape[:2]
            img = img.astype("float32") / 255.0
            mask = mask.astype("float32") / 255.0

            # Generate coordinate grid (x, y)
            x = torch.linspace(-1, 1, w)
            y = torch.linspace(-1, 1, h)
            grid_y, grid_x = torch.meshgrid(y, x, indexing="ij")
            coords = torch.stack([grid_x, grid_y], dim=-1).reshape(h * w, 2)

            # Calculate time value (t_value)
            t_value = int(os.path.splitext(frame_name)[0])
            t_value = (t_value * 2) / (self.VIDEO_LENGTH - 1)
            t = torch.full((coords.shape[0], 1), t_value, dtype=torch.float32)
            coords = torch.cat([coords, t], dim=-1)

            # Flatten image and mask to per-pixel format
            img = torch.tensor(img).view(-1, 3)
            mask = torch.tensor(mask).view(-1, 1)

            output = {
                "coordinates": coords,  # (x, y, t)
                "rgb": img,  # RGB values of the rectangle
                "mask": mask,  # Mask of the rectangle
            }

        elif self.mode == "val":
            # Get rectangle coordinates
            x1, y1, x2, y2 = (
                self.rect_coords[frame_name]["x1"],
                self.rect_coords[frame_name]["y1"],
                self.rect_coords[frame_name]["x2"],
                self.rect_coords[frame_name]["y2"],
            )

            # Image and mask dimensions
            h, w = x2 - x1, y2 - y1

            # Normalize image and mask
            img = img.astype("float32") / 255.0
            mask = mask.astype("float32") / 255.0

            # Generate coordinate grid (x, y) within the rectangle
            x = torch.linspace(x1 / img.shape[1] * 2 - 1, x2 / img.shape[1] * 2 - 1, w)
            y = torch.linspace(y1 / img.shape[0] * 2 - 1, y2 / img.shape[0] * 2 - 1, h)
            grid_y, grid_x = torch.meshgrid(y, x, indexing="ij")
            coords = torch.stack([grid_x, grid_y], dim=-1).reshape(h * w, 2)

            # Calculate t_value
            t_value = int(os.path.splitext(frame_name)[0])
            t_value = (t_value * 2) / (self.VIDEO_LENGTH - 1)
            t = torch.full((coords.shape[0], 1), t_value, dtype=torch.float32)
            coords = torch.cat([coords, t], dim=-1)

            # Convert image and mask to tensors
            img = torch.tensor(img).view(-1, 3)
            mask = torch.tensor(mask).view(-1, 1)

            output = {
                "coordinates": coords,  # (x, y, t)
                "original_image": img,  # Original image
                "original_mask": mask,  # Original mask
                "rectangle_coords": [x1, y1, x2, y2],  # Rectangle coordinates
            }

        return output

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################


def create_dataloaders(
    dir_name: Path, batch_size: int = 1, num_workers: int = 0
) -> tuple[DataLoader, DataLoader]:
    """
    Creates DataLoader objects for training and validation sets.

    Parameters
    ----------
    dir_name : Path
        Path to the main directory containing the data.
    batch_size : int, optional
        Batch size for the training loader, default 1.
    num_workers : int, optional
        Number of threads used for data loading, default 0.

    Returns
    -------
    tuple[DataLoader, DataLoader]
        Returns a tuple containing the DataLoader for training and validation sets.
    """
    # Path to the training set
    train_path = Path(dir_name / "train")
    frame_names = sorted(os.listdir(train_path / "images"))
    train_dataset = ImageDataset(
        Path(train_path / "images"),
        Path(train_path / "masks"),
        frame_names,
        mode="train",
    )

    # Path to the validation set
    val_path = Path(dir_name / "val")
    frame_names = sorted(os.listdir(val_path / "images"))
    val_dataset = ImageDataset(
        Path(val_path / "images"),
        Path(val_path / "masks"),
        frame_names,
        mode="val",
        json_path=val_path / "rectangles.json",
    )

    # Create DataLoaders
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers
    )
    val_loader = DataLoader(
        val_dataset, batch_size=1, shuffle=False, num_workers=num_workers
    )

    return train_loader, val_loader

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################
tempdir = tempfile.TemporaryDirectory()
TMP_DIR = tempdir.name
DATA_PATH = Path(TMP_DIR) / Path("data")

def unpack_tar_gz(filename: str, path: Path = DATA_PATH) -> None:
    """ Unpacks a tar.gz archive """
    with tarfile.open(filename, "r:gz") as tar:
        tar.extractall(path=path)
    
unpack_tar_gz("./train.tar.gz", DATA_PATH / Path("train"))
unpack_tar_gz("./val.tar.gz", DATA_PATH / Path("val"))
train_loader, val_loader = create_dataloaders(
    DATA_PATH, batch_size=2, num_workers=0
)

### Evaluation Criteria Code

Code similar to the one below will be used to evaluate the solution on the test set.

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################


def grade(
    model: torch.nn.Module,
    data_loader: torch.utils.data.DataLoader,
    device: torch.device = "cuda",
) -> tuple[float, float, float]:
    """Evaluates the model based on data from the DataLoader.

    The function calculates the average PSNR (Peak Signal-to-Noise Ratio) and mask accuracy,
    and returns the final score.

    Parameters
    ----------
    model : torch.nn.Module
        Model to evaluate.
    data_loader : torch.utils.data.DataLoader
        DataLoader containing evaluation data.
    device : torch.device
        Device on which calculations are performed (e.g., 'cuda' or 'cpu').

    Returns
    -------
    tuple[float, float, float]
        Returns a tuple containing:
        - Final points (float)
        - Average PSNR (float)
        - Average accuracy (float)
    """
    model.eval()  # Set model to evaluation mode
    psnr_list, acc_list = [], []  # Lists to store PSNR and accuracy results

    with torch.no_grad():  # Disable gradients for evaluation
        for batch_idx, batch in enumerate(data_loader):
            # Get data from the batch
            coordinates = batch["coordinates"].to(device)
            original_image = (
                batch["original_image"][0].cpu().numpy().reshape(256, 256, 3)
            )
            original_image = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
            original_mask = batch["original_mask"][0].cpu().numpy().reshape(256, 256)
            rect_coords = batch["rectangle_coords"]  # (x1, y1, x2, y2)

            # Send model to GPU
            model = model.cuda()
            rgb_pred, mask_pred = model(coordinates)
            rgb_pred = rgb_pred.squeeze(0).detach().cpu().numpy()
            rgb_pred = rgb_pred[:, [2, 1, 0]]
            mask_pred = mask_pred.squeeze(0).detach().cpu().numpy().squeeze()

            # Create reconstructed images
            inpainted_img = original_image.copy()
            inpainted_mask = original_mask.copy()
            x1, y1, x2, y2 = map(int, rect_coords)
            idx = 0
            for y in range(y1, y2):
                for x in range(x1, x2):
                    inpainted_img[y, x, :] = rgb_pred[idx]
                    inpainted_mask[y, x] = mask_pred[idx]
                    idx += 1

            # Calculate PSNR for the rectangle region
            gt_region = original_image[y1:y2, x1:x2, :]
            pred_region = inpainted_img[y1:y2, x1:x2, :]
            psnr = peak_signal_noise_ratio(gt_region, pred_region, data_range=1.0)
            psnr_list.append(psnr)

            # Calculate accuracy for the mask region
            gt_mask_region = original_mask[y1:y2, x1:x2]
            pred_mask_region = inpainted_mask[y1:y2, x1:x2]
            pred_mask_region = (pred_mask_region > 0.5).astype(np.uint8)
            acc = np.mean(pred_mask_region == gt_mask_region)
            acc_list.append(acc)

            # Visualize results for the first batch
            if batch_idx == 0:
                fig, axs = plt.subplots(1, 3, figsize=(15, 5))
                axs[0].imshow(original_image)
                axs[0].set_title("Original Image")
                axs[1].imshow(inpainted_img)
                axs[1].set_title("Inpainted Image")
                axs[2].imshow(inpainted_mask, cmap="gray")
                axs[2].set_title("Inpainted Mask")
                for ax in axs:
                    ax.axis("off")
                plt.tight_layout()
                plt.show()

    # Calculate average PSNR and accuracy scores
    psnr_score = np.mean(psnr_list)
    acc_score = np.mean(acc_list)

    # Calculate points based on PSNR
    if psnr_score < 15.5:
        p_psnr = 0
    elif psnr_score < 23.5:
        p_psnr = (psnr_score - 15.5) * 5/4
    else:
        p_psnr = 10

    # Calculate points based on accuracy
    if acc_score < 0.83:
        p_acc = 0
    elif acc_score < 0.98:
        p_acc = (acc_score - 0.83) * 20/3
    else:
        p_acc = 1

    # Total points
    points = 7 * p_psnr + 30 * p_acc

    return int(round(points, 0)), psnr_score, acc_score

### Example Solution

Below we present a simplified solution that serves as an example demonstrating the basic functionality of the notebook.

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

class DummyINR(nn.Module):
    """
    DummyINR is a simple neural network model that generates pink RGB values
    and a zero mask based on the given input coordinates.
    """

    def __init__(self):
        """Initializes an instance of the DummyINR class."""
        super(DummyINR, self).__init__()

    def forward(self, coords):
        """
        Calculates pink RGB values and a zero mask based on input coordinates.

        Parameters:
        ----------
        coords : torch.Tensor
            Input tensor of shape (B, N, 3), where B is the batch size,
            N is the number of points, and 3 are the coordinates (x, y, t).

        Returns:
        -------
        tuple:
            Two output values containing:
                - rgb : torch.Tensor
                    Tensor of pink RGB values in range [0, 1] of shape (B, N, 3).
                - mask : torch.Tensor
                    Tensor of zero mask of shape (B, N, 1).
        """
        batch_size, num_points, _ = coords.shape

        pink_rgb = torch.tensor(
            [1, 0, 1], dtype=torch.float32, device=coords.device
        )
        rgb = pink_rgb.unsqueeze(0).unsqueeze(0).expand(batch_size, num_points, -1)  # Shape: (B, N, 3)

        # Mask values in the range [0, 1]
        mask = torch.zeros(
            (batch_size, num_points, 1), dtype=torch.float32, device=coords.device
        )  # Shape: (B, N, 1)

        return rgb, mask

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

if not FINAL_EVALUATION_MODE:
    dummy_model = DummyINR()
    points, psnr, accuracy = grade(dummy_model, val_loader)
    print(f"The example solution achieved: {points} points on the validation set.")
    print(f"PSNR: {psnr:.2f}")
    print(f"Accuracy: {accuracy:.2f}")

## Your Solution

In this section, you should place your solution. Make changes only here!

In [ ]:
class YourSolution(nn.Module):
    """YourSolution class

    The class implements an INR neural network model that processes input
    coordinates (x, y, t) and returns RGB values and a mask.

    Attributes:
    ----------
    No attributes to initialize in the constructor.
    The model should be defined in the `__init__` method.
    """

    def __init__(self):
        """Initializes YourSolution model.

        Here you should define all the layers and components of the model.
        """
        super(YourSolution, self).__init__()
        # Model initialization
        pass

    def forward(self, coords):
        """Processes input coordinates and returns model outputs.

        Parameters:
        ----------
        coords : torch.Tensor
            Input tensor of shape (B, N, 3), where each column
            corresponds to coordinates (x, y, t).

        Returns:
        --------
        tuple
            Two output values:
                - "rgb" : torch.Tensor
                    Tensor with RGB values in range [0, 1].
                - "mask" : torch.Tensor
                    Tensor with mask values in range [0, 1].
        """
        # Model implementation
        pass



In [ ]:
def train(
    model,
    train_loader,
    epochs=1,
    lr=1000,
    device="cuda" if torch.cuda.is_available() else "cpu",
):
    """Function that trains the model on the given dataset.

    Parameters
    ----------
    model : torch.nn.Module
        The model to be trained.
    train_loader : torch.utils.data.DataLoader
        DataLoader containing training data.
    epochs : int, optional
        Number of training epochs (default 1).
    lr : float, optional
        Learning rate (default 1000).
    device : str, optional
        Device on which to train the model (default "cuda" if available).

    Notes
    -----
    The function requires further implementation of the training loop and learning logic.
    """
    # Build the training loop
    model = model.to(device)
    optimizer = torch.optim.Adam(
        model.parameters(), lr=lr
    )  # You can change the optimizer
    pass


In [ ]:
# Enter the number of epochs you want to run and the chosen learning rate
EPOCHS = 40
LEARNING_RATE = 1e-4

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

model = YourSolution()
train(model, train_loader, epochs=EPOCHS, lr=LEARNING_RATE)

## Evaluation

Running the cell below will allow you to check how many points your solution would score on the validation data. Before submitting, make sure that the entire notebook runs from start to finish without errors in *FINAL_EVALUATION_MODE = True* mode and without the need for user intervention after selecting the "Run All" option.

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

if not FINAL_EVALUATION_MODE:
    points, psnr, accuracy = grade(model, val_loader)
    print(f"Your solution achieved: {points} points on the validation set.")
    print(f"PSNR: {psnr:.2f}")
    print(f"Classification accuracy: {accuracy:.2f}%")

During checking, the model will be saved as `your_model.pkl` and evaluated on the test set.

In [ ]:
######################### DO NOT CHANGE THIS CELL ##########################

if FINAL_EVALUATION_MODE:
    import cloudpickle

    OUTPUT_PATH = "file_output"
    FUNCTION_FILENAME = "your_model.pkl"
    FUNCTION_OUTPUT_PATH = os.path.join(OUTPUT_PATH, FUNCTION_FILENAME)

    if not os.path.exists(OUTPUT_PATH):
        os.makedirs(OUTPUT_PATH)

    with open(FUNCTION_OUTPUT_PATH, "wb") as f:
        cloudpickle.dump(model, f)